# 04 — Model Comparison

## Objective

This notebook compares several classification algorithms for customer churn prediction using a consistent preprocessing and validation framework.

The goals are to:

- compare linear, tree-based and ensemble models;
- evaluate models using stratified cross-validation;
- measure both predictive performance and stability across folds;
- avoid using the held-out test set for model selection;
- identify the strongest candidates for subsequent hyperparameter tuning.

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_validate
)

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier
)

from sklearn.metrics import (
    make_scorer,
    precision_score,
    recall_score,
    f1_score
)

In [2]:
PROJECT_ROOT = Path("..")

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "WA_FnUseC_TelcoCustomerChurn.csv"
)

df = pd.read_csv(DATA_PATH)

df.shape

(7043, 21)

In [3]:
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

In [4]:
X = df.drop(
    columns=[
        "customerID",
        "Churn"
    ]
)

y = df["Churn"]

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [6]:
numeric_features = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges"
]

categorical_features = [
    column
    for column in X_train.columns
    if column not in numeric_features
]

In [7]:
numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

In [8]:
categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ]
)

In [9]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_pipeline,
            numeric_features
        ),
        (
            "cat",
            categorical_pipeline,
            categorical_features
        )
    ]
)

## Candidate Models

Four classifiers are compared.

**Logistic Regression** serves as the linear baseline established previously.

**Decision Tree** introduces non-linear decision boundaries and interactions between variables, but individual trees can easily overfit.

**Random Forest** combines many decision trees using bagging and random feature selection, typically reducing the variance of an individual tree.

**Gradient Boosting** builds trees sequentially, with each new estimator focusing on correcting errors made by the previous ensemble.

In [10]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        random_state=42
    ),

    "Decision Tree": DecisionTreeClassifier(
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ),

    "Gradient Boosting": GradientBoostingClassifier(
        random_state=42
    )
}

## Cross-Validation Setup

All candidate models are evaluated using the same stratified 5-fold cross-validation strategy.

The complete preprocessing workflow is included inside each model pipeline, ensuring that preprocessing parameters are learned independently within each training fold.

The held-out test set is not used during model comparison.

In [11]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [12]:
scoring = {
    "accuracy": "accuracy",
    "precision": make_scorer(
        precision_score,
        pos_label="Yes",
        zero_division=0
    ),
    "recall": make_scorer(
        recall_score,
        pos_label="Yes",
        zero_division=0
    ),
    "f1": make_scorer(
        f1_score,
        pos_label="Yes",
        zero_division=0
    ),
    "roc_auc": "roc_auc"
}

In [14]:
comparison_results = []
cv_results_by_model = {}

for model_name, classifier in models.items():

    model_pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("classifier", classifier)
        ]
    )

    results = cross_validate(
        model_pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        return_train_score=True,
        n_jobs=-1
    )

    cv_results_by_model[model_name] = results

    comparison_results.append({
        "model": model_name,

        "accuracy_mean": results["test_accuracy"].mean(),
        "accuracy_std": results["test_accuracy"].std(),

        "precision_mean": results["test_precision"].mean(),
        "precision_std": results["test_precision"].std(),

        "recall_mean": results["test_recall"].mean(),
        "recall_std": results["test_recall"].std(),

        "f1_mean": results["test_f1"].mean(),
        "f1_std": results["test_f1"].std(),

        "roc_auc_mean": results["test_roc_auc"].mean(),
        "roc_auc_std": results["test_roc_auc"].std()
    })

In [15]:
model_comparison = pd.DataFrame(comparison_results)

model_comparison = model_comparison.sort_values(
    by="roc_auc_mean",
    ascending=False
).reset_index(drop=True)

model_comparison.round(4)

,model,accuracy_mean,accuracy_std,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,roc_auc_mean,roc_auc_std
0,Gradient Boosting,0.8033,0.0108,0.6606,0.0284,0.5338,0.0234,0.5902,0.0223,0.8478,0.0121
1,Logistic Regression,0.8026,0.0120,0.6547,0.0280,0.5431,0.0411,0.5930,0.0298,0.8461,0.0125
2,Random Forest,0.7861,0.0108,0.6290,0.0322,0.4763,0.0220,0.5416,0.0207,0.8200,0.0118
3,Decision Tree,0.7299,0.0040,0.4913,0.0074,0.4977,0.0240,0.4941,0.0111,0.6559,0.0078


In [16]:
overfitting_results = []

for model_name, results in cv_results_by_model.items():

    train_roc_auc = results["train_roc_auc"].mean()
    validation_roc_auc = results["test_roc_auc"].mean()

    overfitting_results.append({
        "model": model_name,
        "train_roc_auc": train_roc_auc,
        "validation_roc_auc": validation_roc_auc,
        "gap": train_roc_auc - validation_roc_auc
    })

overfitting_comparison = pd.DataFrame(
    overfitting_results
).sort_values(
    by="gap",
    ascending=False
).reset_index(drop=True)

overfitting_comparison.round(4)

,model,train_roc_auc,validation_roc_auc,gap
0,Decision Tree,1.0000,0.6559,0.3441
1,Random Forest,0.9999,0.8200,0.1799
2,Gradient Boosting,0.8878,0.8478,0.0400
3,Logistic Regression,0.8496,0.8461,0.0034


## Initial Model Comparison

The candidate models are compared using identical preprocessing and stratified cross-validation folds.

Model selection should consider both average predictive performance and variability across folds.

The difference between training and validation ROC-AUC is also inspected as an indicator of potential overfitting. A model with extremely high training performance but substantially weaker validation performance may be learning patterns that do not generalize well to unseen customers.

The held-out test set remains untouched during this comparison.